# SaaS Market 2026 — Exploratory Data Analysis

This notebook explores the **330-tool SaaS market dataset** after the pipeline has run.

Run `python src/pipeline.py` first to generate the processed Parquet files.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils.config import CLEAN_PARQUET, MART_CATEGORY, MART_PRICING, MART_FEATURES
from src.utils.io import read_parquet

# Load data
df    = read_parquet(CLEAN_PARQUET)
mc    = read_parquet(MART_CATEGORY)
mp    = read_parquet(MART_PRICING)
mf    = read_parquet(MART_FEATURES)

print(f'Clean dataset: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(3)

## 1. Dataset Overview

In [ ]:
print('=== Basic stats ===')
print(f'  Tools        : {len(df)}')
print(f'  Categories   : {df["category"].nunique()}')
print(f'  Verticals    : {list(df["vertical"].unique())}')
print(f'  Free plans   : {df["free_plan"].mean()*100:.1f}%')
print(f'  Avg rating   : {df["rating"].mean():.3f}')
print(f'  Avg features : {df["features_count"].mean():.1f}')
print()
df.describe().round(2)

## 2. Distribution by Vertical

In [ ]:
vert_counts = df['vertical'].value_counts().reset_index()
fig = px.pie(
    vert_counts, values='count', names='vertical',
    title='Tool distribution by vertical',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 3. Top 15 Categories by Tool Count

In [ ]:
top15 = mc.head(15).sort_values('tool_count')
fig = px.bar(
    top15, x='tool_count', y='category', orientation='h',
    color='avg_rating', color_continuous_scale='RdYlGn',
    title='Top 15 categories — tool count & avg rating',
    labels={'tool_count': 'Number of tools', 'category': 'Category', 'avg_rating': 'Avg rating'}
)
fig.update_layout(height=500)
fig.show()

## 4. Pricing Distribution

In [ ]:
# Filter extreme outliers for readability
df_plot = df[df['starting_price_usd'] <= 200]

fig = px.histogram(
    df_plot, x='starting_price_usd', color='vertical', nbins=40,
    barmode='overlay', opacity=0.75,
    title='Starting price distribution (≤ $200/mo)',
    labels={'starting_price_usd': 'Starting price (USD/mo)'}
)
fig.show()

In [ ]:
# Pricing tier breakdown per vertical
tier_order = ['Free', 'Budget', 'Mid-Market', 'Premium']
mp['pricing_tier'] = pd.Categorical(mp['pricing_tier'], categories=tier_order, ordered=True)
mp_sorted = mp.sort_values(['vertical', 'pricing_tier'])

fig = px.bar(
    mp_sorted, x='vertical', y='share_pct', color='pricing_tier',
    title='Pricing tier share per vertical (%)',
    labels={'share_pct': 'Share (%)', 'vertical': 'Vertical', 'pricing_tier': 'Tier'},
    color_discrete_sequence=px.colors.sequential.Viridis_r
)
fig.update_layout(barmode='stack')
fig.show()

## 5. Rating Analysis

In [ ]:
fig = px.box(
    df, x='vertical', y='rating', color='vertical',
    title='Rating distribution by vertical',
    points='all', color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.show()

## 6. Features vs. Rating Scatter

In [ ]:
fig = px.scatter(
    df, x='features_count', y='rating',
    color='vertical', size='starting_price_usd',
    size_max=30, opacity=0.7,
    hover_name='tool_name',
    title='Features count vs. Rating (bubble = starting price)',
    trendline='ols',
    labels={'features_count': 'Number of features', 'rating': 'User rating'}
)
fig.show()

## 7. Category-level Correlation: Features vs. Rating

In [ ]:
mf_plot = mf[mf['tool_count'] >= 5]  # only categories with enough data

fig = px.scatter(
    mf_plot, x='avg_features', y='avg_rating',
    size='tool_count', text='category',
    title='Avg features vs. avg rating per category (n≥5)',
    labels={'avg_features': 'Avg features', 'avg_rating': 'Avg rating'},
    color='corr_features_rating', color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0
)
fig.update_traces(textposition='top center', textfont_size=10)
fig.update_layout(height=550)
fig.show()

## 8. Free vs. Paid Plan Comparison

In [ ]:
free_vs_paid = df.groupby('free_plan').agg(
    count=('tool_name', 'count'),
    avg_rating=('rating', 'mean'),
    avg_features=('features_count', 'mean'),
    avg_plan_count=('plan_count', 'mean'),
).reset_index()
free_vs_paid['free_plan'] = free_vs_paid['free_plan'].map({True: 'Has free plan', False: 'Paid only'})

fig = make_subplots(rows=1, cols=3, subplot_titles=['Avg Rating', 'Avg Features', 'Avg Plan Count'])

colors = ['#2ecc71', '#e74c3c']
for i, metric in enumerate(['avg_rating', 'avg_features', 'avg_plan_count'], start=1):
    fig.add_trace(
        go.Bar(x=free_vs_paid['free_plan'], y=free_vs_paid[metric],
               marker_color=colors, showlegend=False),
        row=1, col=i
    )

fig.update_layout(title_text='Free plan vs. Paid-only — key metrics comparison', height=400)
fig.show()

## 9. Price Range Heatmap by Category & Vertical

In [ ]:
pivot = df.pivot_table(
    index='category', columns='vertical',
    values='highest_plan_price_usd', aggfunc='median'
).fillna(0).round(0)

fig = px.imshow(
    pivot, text_auto=True, aspect='auto',
    color_continuous_scale='Blues',
    title='Median highest plan price (USD) — category × vertical'
)
fig.update_layout(height=700)
fig.show()

## 10. mart_category Summary Table

In [ ]:
mc.style \
  .background_gradient(subset=['tool_count'], cmap='Blues') \
  .background_gradient(subset=['avg_rating'], cmap='RdYlGn') \
  .format({'free_plan_pct': '{:.1f}%', 'avg_rating': '{:.3f}',
           'median_starting_price': '${:.2f}', 'median_highest_price': '${:.2f}'}) \
  .set_caption('mart_category — one row per category')